# `bloqify` Syntactic Sugar

This notebook shows how we can use syntactic sugar, in particular the `bloqify` decorator to quickly compose bloqs in Qualtran.

In [ ]:
# Top-level qualtran imports
import qualtran as qlt
import qualtran.dtype as qdt

## Example: Negate

We'll use the overloaded operators on quantum variables as an introductory example of how to define a quantum subroutine using syntactic sugar.

 - We define an ordinary function that takes in the quantum variable `x`. This function can take arbitrary classical compile-time parameters as well.
 - The `bloqify` decorator lets us record calls to this function as quantum subroutine calls.
 - The function takes a bloq-builder as its first argument. This is required. It's usually helpful to pass along if you need to allocate new quantum variables. Here, it's unused.
 - The `~` operator performs a bitwise-not via `qualtran.bloqs.arithmetic.BitwiseNot`.
 - The `+=` operator with a compile-time classical right-hand argument performs a quantum-classical addition via `qualtran.bloqs.arithmetic.AddK`.
 - A bloqify function returns a string-keyed dictionary. The keys set the output register names.

In [ ]:
@qlt.bloqify
def negate(bb, x):
    x = ~x
    x += 1
    return {'x': x}

### Using our `negate` function

You can "call" a `bloqify` function from another quantum function definition: a caller `bloqify` function; a `build_composite_bloq` method override; or a raw `BloqBuilder` construction (as below). When you "call" the bloqify function, the system will record it as a quantum subroutine (analogous to a call to `bb.add(bloq, ...)`.

In [ ]:
# Set up a simple, outer, "calling" function definition
bb = qlt.BloqBuilder()
x = bb.alloc_qint(bitsize=8, k=5)

# Perform a quantum call to our bloqify function (!)
x = negate(bb, x)

# Finish our outer function definition
program = bb.finalize(x_out=x)

In [ ]:
program.draw()

In [ ]:
program.call_classically()

### Instantiating `negate`.

In (ordinary) Qualtran, we're used to instantiating bloqs as Python objects. If you'd like a bloq object from a `bloqify` function, you must provide additional information, namely the quantum signature specifying the input and output quantum data types.

- `qlt.Bloq` objects declare their quantum `qlt.Signature` a priori. `bloqify` functions infer their signatures when they're "called". To instantiate a `qlt.Bloq` object from a `bloqify` function, you must explicitly pass a signature to `bloqify_func.make(...)`.

In [ ]:
negate_bloq = negate.make(qlt.qsig(x=qdt.QInt(8)))
negate_bloq.draw(type='musical_score')

negate_bloq = negate.make(qlt.qsig(x=qdt.QInt(64)))
negate_bloq.draw(type='musical_score')

In [ ]:
print('x_out =', negate_bloq.call_classically(x=5))

### Dumping `negate` to L1

In [ ]:
negate.print_l1(qlt.qsig(x=qdt.QInt(8)))

## Example: Making a Bell state

As a second example, we'll create a simple program that allocates a bell state by allocating two qubits and performing the appropriate 1- and 2-qubit operations.

 - Again, we use the `bloqify` decorator so we can call this from other quantum functions.
 - It takes a `bb: BloqBuidler` as its only argument since this function allocates a new state rather than operating on any existing quantum variables.
 - We use methods on `BloqBuilder` to do common atomic operations like `bb.H(q)` to call the Hadamard gate on quantum variable `q`.

In [ ]:
@qlt.bloqify
def bell(bb):
    q0 = bb.alloc_qbit()
    q0 = bb.H(q0)

    q1 = bb.alloc_qbit()
    q0, q1 = bb.CNOT(q0, q1)
    return {'q0': q0, 'q1': q1}

In [ ]:
bell.draw(type='musical_score')

### Instantiating `bell`

To recover a Python `qlt.Bloq` object, we again call `bloqify_func.make(...)`. Since this function does not take any quantum inputs or classical compile-time parameters, we can call `make` with no arguments.

In [ ]:
bell_bloq = bell.make()
print(repr(bell_bloq))

When we have the bloq object, we can use any of the Qualtran analysis routines that operates on bloqs. Here, we perform a numerical simulation (tensor contraction) to get the state vector representation of the simple circuit. It is indeed a Bell state.

In [ ]:
bell.make().tensor_contract()

## Building `negate` from the ground up

When we used the operators `~` and `+=`, we used bloqs from the standard library (namely `BitwiseNot` and `AddK`) to build our program. For illustrative purposes, we'll implement more of the callees of a `negate` subroutine from a simpler instruction set. We'll write our routines in terms:

 - atomic bloqs (`CNOT`, `X`, ...)
 - `qualtran.bloqs.arithmetic.Add` quantum-quantum adder


### `xor_k`

This bloq does $x \oplus k$ for quantum $x$ and classical $k$. Specifically: it flips bits in `x` according to the bit-encoding of the classical compile-time integer `k`.

 - We use `x[:]` to split e.g. a quantum integer into its individual qubits.
 - We use `x.dtype` to write generic code without boilerplate. Here, we use `x.dtype` to turn the classical integer `k` into bits.
 - We use the `~q` operator on individual qubits (i.e. `QVar` with `dtype=QBit()`). The operator knows to use `qualtran.bloqs.basic_gates.XGate`.

In [ ]:
@qlt.bloqify
def xor_k(bb: 'qlt.BloqBuilder', x: 'qlt.QVar', k: int):
    """XOR the classical value `k` into the `x` register."""
    xs = x[:]
    for i, bit in enumerate(x.dtype.to_bits(k)):
        if bit == 1:
            xs[i] = ~xs[i]

    return {'x': bb.join(xs, dtype=x.dtype)}

### `bitwise_not`

This bloq does $X$ to each qubit.

 - We implicitly split the qubits. The `x` qvar will call split once, and subsequent index access will give you the results of the split operation.
 - We could have used `~` again, but library functions can also be provided by little wrapper functions accessible on `bb`, here: `bb.X`. These just delegate to `XGate.qcall(...)`, which we'll discuss later; but they look nicer.

In [ ]:
@qlt.bloqify
def bitwise_not(bb: 'qlt.BloqBuilder', x: 'qlt.QVar'):
    """NOT (X gate) on each bit encoding `x`"""
    outs = []
    for i in range(len(x)):
        out = bb.X(x[i])
        outs.append(out)

    return {'x': bb.join(outs)}

### `add_k`

This bloq adds a constant by loading it into a register with clifford operations and performing a quantum-quantum addition.

 - We use our previously-defined `xor_k` function. You can see that you call it like normal.
 - We're using `Add` from the standard library. Bloqs can learn a new classmethod `qcall` which takes both quantum variables and classical compile-time parameters to instantiate a bloq object and add it to the program. Here, `Add.qcall` knows how to deduce the two data types from the quantum variables; and it can extract the `bb` object from within the quantum variables.

In [ ]:
@qlt.bloqify
def add_k(bb: 'qlt.BloqBuilder', x: 'qlt.QVar', k: int):
    """A classical `k` into quantum `x`."""
    from qualtran.bloqs.arithmetic import Add

    # load `k`
    qk = bb.allocate(dtype=x.dtype)
    qk = xor_k(bb, x=qk, k=k)

    # Add
    qk, x = Add.qcall(a=qk, b=x)

    # unload `k`
    qk = xor_k(bb, x=qk, k=k)
    bb.free(qk)

    return {'x': x}

### Negate

We put things together in the same way, calling our functions we defined above.

In [ ]:
@qlt.bloqify
def negate(bb: 'qlt.BloqBuilder', x: 'qlt.QVar'):
    x = bitwise_not(bb, x=x)
    x = add_k(bb, x=x, k=1)
    return {'x': x}

### Instantiating

This is the same as before. To get a bloq object, we need to give the overall signature, while all subsequent callees deduce data types from the quantum variables that get passed from callers.

In [ ]:
bloq = negate.make(qlt.Signature.build(x=qdt.QInt(4)))
qlt.show_bloq(bloq)

### Complete L1 QLT Intermediate Representation (IR)

We can compile our entire program to the QLT intermediate representation (IR) to view the full, static program.

In [ ]:
def is_in_isa(bloq: qlt.Bloq):
    from qualtran.bloqs.bookkeeping._bookkeeping_bloq import _BookkeepingBloq
    from qualtran.bloqs.basic_gates import XGate
    from qualtran.bloqs.arithmetic import Add

    return isinstance(bloq, (_BookkeepingBloq, XGate, Add))

In [ ]:
from qualtran.l1 import dump_l1
print(dump_l1(bloq, force_extern_pred=is_in_isa))

### Call graph

We can inspect the call graph of bloqs defined via `@bloqify`.

Whereas the built-in `AddK` bloq can override its callgraph, declare its tensors, and be queried for its compile-time classical parameters; these bloq objects are all named instantiations of the `CompositeBloq` container class, which can be limiting. To promote
an L3 `@bloq` function to its own `Bloq` class, I'll show that somewhere else.

In [ ]:
from qualtran.drawing import show_call_graph
show_call_graph(bloq, agg_gate_counts='t_and_ccz')